# LambdaMART Model Training com SVM-Rank Data

Este notebook demonstra como usar o projeto LambdaMART para:
- Carregar dados no formato SVM-Rank (.txt)
- Treinar um modelo LambdaMART 
- Avaliar com métricas NDCG@1,3,5,10 e MRR
- Registrar resultados no MLflow

## Estrutura do Pipeline

1. **Configuração do Ambiente**: Imports e configurações
2. **MLflow Setup**: Configuração do tracking de experimentos
3. **Carregamento de Dados**: Leitura de arquivos SVM-Rank
4. **Preprocessamento**: Preparação dos dados para treinamento
5. **Configuração de Hiperparâmetros**: Definição dos parâmetros do modelo
6. **Treinamento**: Treinamento do modelo LambdaMART
7. **Predição**: Geração de predições no conjunto de teste
8. **Avaliação**: Cálculo das métricas de ranking
9. **MLflow Logging**: Registro de experimentos e artefatos
10. **Visualização**: Análise e visualização dos resultados

## 1. Configuração do Ambiente e Imports

In [ ]:
# Imports das bibliotecas necessárias
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
import mlflow
import mlflow.lightgbm
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Adicionar o diretório src ao path para importar nossos módulos
sys.path.append('../src')

# Imports dos nossos módulos personalizados
from data_loader import SVMRankDataLoader
from model import LambdaMARTModel
from evaluator import RankingEvaluator
from mlflow_utils import MLflowManager

# Configurações de visualização
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

print("✅ Todas as bibliotecas importadas com sucesso!")
print(f"LightGBM version: {lgb.__version__}")
print(f"MLflow version: {mlflow.__version__}")

## 2. Configuração do MLflow

In [ ]:
# Configuração do MLflow para tracking de experimentos
experiment_name = "LambdaMART_Notebook_Demo"

# Inicializar o MLflow Manager
mlflow_manager = MLflowManager(
    experiment_name=experiment_name,
    tracking_uri="file:../mlruns"  # Usar tracking local
)

print(f"✅ MLflow configurado com experimento: {experiment_name}")
print(f"📁 Tracking URI: {mlflow.get_tracking_uri()}")

# Verificar se há experimentos anteriores
try:
    runs_df = mlflow_manager.get_experiment_runs()
    if not runs_df.empty:
        print(f"📊 Encontrados {len(runs_df)} runs anteriores no experimento")
        print("Últimos 3 runs:")
        print(runs_df[['run_name', 'status', 'start_time']].head(3))
    else:
        print("🆕 Este será o primeiro run do experimento")
except:
    print("🆕 Novo experimento criado")

## 3. Carregamento e Preparação dos Dados

In [ ]:
# Inicializar o data loader
data_loader = SVMRankDataLoader()

# Definir diretórios dos dados
train_dir = "../data/train"
test_dir = "../data/test"

# Verificar se os diretórios existem, senão criar dados de exemplo
if not os.path.exists(train_dir) or not os.path.exists(test_dir):
    print("📁 Diretórios de dados não encontrados. Criando dados de exemplo...")
    
    # Criar dados de exemplo
    os.makedirs(train_dir, exist_ok=True)
    os.makedirs(test_dir, exist_ok=True)
    
    # Criar dados SVM-Rank de exemplo
    data_loader.create_sample_data("../data", n_queries=30, n_docs_per_query=20)
    
    # Mover arquivos para os diretórios corretos
    import shutil
    for filename in os.listdir("../data"):
        if filename.endswith(".txt"):
            if "train" in filename:
                shutil.move(f"../data/{filename}", train_dir)
            elif "test" in filename:
                shutil.move(f"../data/{filename}", test_dir)
    
    print("✅ Dados de exemplo criados com sucesso!")

# Listar arquivos disponíveis
print("📂 Arquivos de treino:")
train_files = [f for f in os.listdir(train_dir) if f.endswith('.txt')]
for f in train_files:
    print(f"  - {f}")

print("📂 Arquivos de teste:")
test_files = [f for f in os.listdir(test_dir) if f.endswith('.txt')]
for f in test_files:
    print(f"  - {f}")

In [ ]:
# Carregar os dados de treino e teste
print("🔄 Carregando dados...")
data = data_loader.load_train_test_data(train_dir, test_dir)

# Extrair dados de treino
train_features = data['train']['features']
train_labels = data['train']['labels']
train_qids = data['train']['query_ids']

# Extrair dados de teste
test_features = data['test']['features']
test_labels = data['test']['labels']
test_qids = data['test']['query_ids']

print("✅ Dados carregados com sucesso!")
print(f"📊 Estatísticas dos dados:")
print(f"  Treino: {train_features.shape[0]} amostras, {train_features.shape[1]} features, {len(np.unique(train_qids))} queries")
print(f"  Teste:  {test_features.shape[0]} amostras, {test_features.shape[1]} features, {len(np.unique(test_qids))} queries")

## 4. Preprocessamento dos Dados SVM-Rank

In [ ]:
# Análise exploratória dos dados
print("📈 Análise Exploratória dos Dados")
print("=" * 50)

# Distribuição de relevâncias
print("🎯 Distribuição de Relevâncias:")
train_relevance_dist = pd.Series(train_labels).value_counts().sort_index()
test_relevance_dist = pd.Series(test_labels).value_counts().sort_index()

print("Treino:", dict(train_relevance_dist))
print("Teste: ", dict(test_relevance_dist))

# Número de documentos por query
train_docs_per_query = pd.Series(train_qids).value_counts()
test_docs_per_query = pd.Series(test_qids).value_counts()

print(f"\n📄 Documentos por Query:")
print(f"Treino - Média: {train_docs_per_query.mean():.1f}, Min: {train_docs_per_query.min()}, Max: {train_docs_per_query.max()}")
print(f"Teste  - Média: {test_docs_per_query.mean():.1f}, Min: {test_docs_per_query.min()}, Max: {test_docs_per_query.max()}")

# Visualização das distribuições
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Distribuição de relevâncias
axes[0,0].bar(train_relevance_dist.index, train_relevance_dist.values, alpha=0.7, label='Treino')
axes[0,0].bar(test_relevance_dist.index, test_relevance_dist.values, alpha=0.7, label='Teste')
axes[0,0].set_title('Distribuição de Relevâncias')
axes[0,0].set_xlabel('Relevância')
axes[0,0].set_ylabel('Frequência')
axes[0,0].legend()

# Documentos por query
axes[0,1].hist(train_docs_per_query.values, bins=20, alpha=0.7, label='Treino')
axes[0,1].hist(test_docs_per_query.values, bins=20, alpha=0.7, label='Teste')
axes[0,1].set_title('Documentos por Query')
axes[0,1].set_xlabel('Número de Documentos')
axes[0,1].set_ylabel('Frequência')
axes[0,1].legend()

# Estatísticas das features (primeiras 5)
feature_stats = pd.DataFrame(train_features[:, :5]).describe()
axes[1,0].imshow(feature_stats.T, cmap='viridis', aspect='auto')
axes[1,0].set_title('Estatísticas das Features (Top 5)')
axes[1,0].set_xlabel('Estatística')
axes[1,0].set_ylabel('Feature')

# Correlação entre features (primeiras 10)
if train_features.shape[1] >= 10:
    corr_matrix = np.corrcoef(train_features[:, :10].T)
    im = axes[1,1].imshow(corr_matrix, cmap='coolwarm', vmin=-1, vmax=1)
    axes[1,1].set_title('Correlação entre Features (Top 10)')
    plt.colorbar(im, ax=axes[1,1])

plt.tight_layout()
plt.show()

# Preparar informações dos datasets para MLflow
dataset_info = {
    'train': {
        'n_samples': len(train_features),
        'n_features': train_features.shape[1],
        'n_queries': len(np.unique(train_qids)),
        'relevance_levels': len(np.unique(train_labels))
    },
    'test': {
        'n_samples': len(test_features),
        'n_features': test_features.shape[1], 
        'n_queries': len(np.unique(test_qids)),
        'relevance_levels': len(np.unique(test_labels))
    }
}

print(f"\n✅ Preprocessamento concluído!")
print(f"📊 Informações dos datasets preparadas para logging")

## 5. Configuração dos Hiperparâmetros

In [ ]:
# Configuração dos hiperparâmetros do modelo LambdaMART
model_params = {
    'objective': 'lambdarank',           # Objetivo de ranking
    'metric': 'ndcg',                   # Métrica para otimização
    'boosting_type': 'gbdt',            # Gradient Boosting Decision Tree
    'num_leaves': 31,                   # Número de folhas por árvore
    'learning_rate': 0.1,               # Taxa de aprendizado
    'feature_fraction': 0.9,            # Fração de features por árvore
    'bagging_fraction': 0.8,            # Fração de dados para bagging
    'bagging_freq': 5,                  # Frequência do bagging
    'verbose': 1,                       # Verbosidade
    'num_threads': -1,                  # Usar todos os cores
    'ndcg_eval_at': [1, 3, 5, 10],     # Posições para NDCG
    'lambdarank_truncation_level': 10,  # Truncamento LambdaRank
    'min_data_in_leaf': 20,            # Mínimo de dados por folha
    'lambda_l1': 0.0,                  # Regularização L1
    'lambda_l2': 0.0,                  # Regularização L2
    'max_depth': -1,                   # Profundidade máxima
    'seed': 42                         # Seed para reprodutibilidade
}

# Configurações de treinamento
training_config = {
    'num_boost_round': 100,             # Número de rounds de boosting
    'early_stopping_rounds': 15,       # Parada antecipada
    'validation_split': 0.2            # Fração para validação
}

# Configurações de avaliação
evaluation_config = {
    'k_values': [1, 3, 5, 10]          # Valores de k para NDCG@k
}

print("⚙️ Hiperparâmetros Configurados:")
print("=" * 50)

print("🎯 Parâmetros do Modelo:")
for key, value in model_params.items():
    print(f"  {key}: {value}")

print(f"\n🚀 Configurações de Treinamento:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

print(f"\n📊 Configurações de Avaliação:")
for key, value in evaluation_config.items():
    print(f"  {key}: {value}")

# Criar DataFrame com hiperparâmetros para visualização
params_df = pd.DataFrame(list(model_params.items()), columns=['Parâmetro', 'Valor'])
print(f"\n📋 Resumo dos Hiperparâmetros:")
print(params_df.to_string(index=False))

## 6. Treinamento do Modelo LambdaMART

In [ ]:
# Dividir dados de treino para validação
def split_data_by_queries(features, labels, query_ids, validation_split=0.2):
    """Divide os dados por queries para validação."""
    unique_queries = np.unique(query_ids)
    np.random.seed(42)
    np.random.shuffle(unique_queries)
    
    n_val_queries = int(len(unique_queries) * validation_split)
    val_queries = unique_queries[:n_val_queries]
    train_queries = unique_queries[n_val_queries:]
    
    train_mask = np.isin(query_ids, train_queries)
    val_mask = np.isin(query_ids, val_queries)
    
    return (features[train_mask], labels[train_mask], query_ids[train_mask],
            features[val_mask], labels[val_mask], query_ids[val_mask])

# Dividir dados de treino
train_feat, train_lab, train_q, val_feat, val_lab, val_q = split_data_by_queries(
    train_features, train_labels, train_qids, 
    validation_split=training_config['validation_split']
)

print(f"🔄 Dados divididos para treinamento:")
print(f"  Treino:    {len(train_feat)} amostras, {len(np.unique(train_q))} queries")
print(f"  Validação: {len(val_feat)} amostras, {len(np.unique(val_q))} queries")

# Inicializar o modelo
print(f"\n🤖 Inicializando modelo LambdaMART...")
model = LambdaMARTModel(model_params)

# Treinar o modelo
print(f"🚀 Iniciando treinamento...")
start_time = datetime.now()

training_results = model.train(
    train_feat, train_lab, train_q,                    # Dados de treino
    val_feat, val_lab, val_q,                          # Dados de validação
    num_boost_round=training_config['num_boost_round'],
    early_stopping_rounds=training_config['early_stopping_rounds']
)

end_time = datetime.now()
training_duration = (end_time - start_time).total_seconds()

print(f"✅ Treinamento concluído!")
print(f"⏱️ Tempo de treinamento: {training_duration:.2f} segundos")
print(f"🎯 Melhor iteração: {training_results['best_iteration']}")

# Obter informações do modelo
model_summary = model.get_model_summary()
print(f"\n📋 Resumo do Modelo:")
print(f"  Número de árvores: {model_summary['num_trees']}")
print(f"  Número de features: {model_summary['num_features']}")
print(f"  Melhor iteração: {model_summary['best_iteration']}")

## 7. Predição no Conjunto de Teste

In [ ]:
# Gerar predições para o conjunto de teste
print("🔮 Gerando predições para o conjunto de teste...")

test_predictions = model.predict(test_features)
val_predictions = model.predict(val_feat)

print(f"✅ Predições geradas!")
print(f"  Teste: {len(test_predictions)} predições")
print(f"  Validação: {len(val_predictions)} predições")

# Organizar predições por query
test_predictions_by_query = model.predict_with_query_groups(test_features, test_qids)
val_predictions_by_query = model.predict_with_query_groups(val_feat, val_q)

print(f"\n📊 Estatísticas das Predições:")
print(f"  Teste - Min: {test_predictions.min():.4f}, Max: {test_predictions.max():.4f}, Média: {test_predictions.mean():.4f}")
print(f"  Validação - Min: {val_predictions.min():.4f}, Max: {val_predictions.max():.4f}, Média: {val_predictions.mean():.4f}")

# Visualizar distribuição das predições
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Distribuição das predições
axes[0].hist(test_predictions, bins=30, alpha=0.7, label='Teste', density=True)
axes[0].hist(val_predictions, bins=30, alpha=0.7, label='Validação', density=True)
axes[0].set_title('Distribuição das Predições')
axes[0].set_xlabel('Score Predito')
axes[0].set_ylabel('Densidade')
axes[0].legend()

# Predições vs Labels Verdadeiros (Teste)
axes[1].scatter(test_labels, test_predictions, alpha=0.6, s=10)
axes[1].set_title('Predições vs Labels Verdadeiros (Teste)')
axes[1].set_xlabel('Label Verdadeiro')
axes[1].set_ylabel('Predição')
axes[1].plot([test_labels.min(), test_labels.max()], [test_labels.min(), test_labels.max()], 'r--', alpha=0.8)

# Boxplot das predições por nível de relevância
prediction_by_relevance = []
relevance_levels = []
for rel_level in sorted(np.unique(test_labels)):
    mask = test_labels == rel_level
    if np.sum(mask) > 0:
        prediction_by_relevance.append(test_predictions[mask])
        relevance_levels.append(f'Rel {int(rel_level)}')

axes[2].boxplot(prediction_by_relevance, labels=relevance_levels)
axes[2].set_title('Distribuição de Predições por Relevância')
axes[2].set_xlabel('Nível de Relevância')
axes[2].set_ylabel('Predição')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

# Exemplo de ranking para uma query específica
sample_qid = test_qids[0]
sample_mask = test_qids == sample_qid
sample_labels = test_labels[sample_mask]
sample_preds = test_predictions[sample_mask]

# Ordenar por predições (ranking)
sorted_indices = np.argsort(sample_preds)[::-1]
sorted_labels = sample_labels[sorted_indices]
sorted_preds = sample_preds[sorted_indices]

print(f"\n🔍 Exemplo de Ranking para Query {sample_qid}:")
print("Posição | Label Verdadeiro | Predição")
print("-" * 40)
for i, (label, pred) in enumerate(zip(sorted_labels[:10], sorted_preds[:10])):
    print(f"{i+1:7d} | {label:15.0f} | {pred:8.4f}")

## 8. Cálculo das Métricas de Avaliação

In [ ]:
# Inicializar o avaliador
evaluator = RankingEvaluator()

# Avaliar o modelo no conjunto de teste
print("📊 Calculando métricas de avaliação...")

test_metrics = evaluator.evaluate_ranking(
    test_labels, test_predictions, test_qids,
    k_values=evaluation_config['k_values']
)

# Avaliar no conjunto de validação
val_metrics = evaluator.evaluate_ranking(
    val_lab, val_predictions, val_q,
    k_values=evaluation_config['k_values']
)

print("✅ Métricas calculadas com sucesso!")

# Exibir resultados detalhados
print("\n" + "="*60)
print("📈 RESULTADOS DE AVALIAÇÃO")
print("="*60)

print("\n🎯 CONJUNTO DE TESTE:")
print("-" * 30)
for metric, value in test_metrics.items():
    if metric.startswith('ndcg'):
        print(f"{metric.upper():<12}: {value:.4f}")
    else:
        print(f"{metric.upper():<12}: {value:.4f}")

print("\n🎯 CONJUNTO DE VALIDAÇÃO:")
print("-" * 30)
for metric, value in val_metrics.items():
    if metric.startswith('ndcg'):
        print(f"{metric.upper():<12}: {value:.4f}")
    else:
        print(f"{metric.upper():<12}: {value:.4f}")

# Obter avaliação detalhada (por query)
detailed_test_eval = evaluator.get_detailed_evaluation(
    test_labels, test_predictions, test_qids,
    k_values=evaluation_config['k_values']
)

print(f"\n📊 Queries Avaliadas:")
print(f"  Teste: {detailed_test_eval['num_valid_queries']}/{detailed_test_eval['num_total_queries']} queries com documentos relevantes")

# Criar DataFrame com métricas para comparação
metrics_comparison = pd.DataFrame({
    'Teste': test_metrics,
    'Validação': val_metrics
})

print(f"\n📋 Comparação de Métricas:")
print(metrics_comparison.round(4))

# Visualizar métricas
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Gráfico de barras das métricas NDCG
ndcg_metrics = [k for k in test_metrics.keys() if k.startswith('ndcg')]
test_ndcg_values = [test_metrics[k] for k in ndcg_metrics]
val_ndcg_values = [val_metrics[k] for k in ndcg_metrics]

x = np.arange(len(ndcg_metrics))
width = 0.35

axes[0].bar(x - width/2, test_ndcg_values, width, label='Teste', alpha=0.8)
axes[0].bar(x + width/2, val_ndcg_values, width, label='Validação', alpha=0.8)
axes[0].set_xlabel('Métrica NDCG')
axes[0].set_ylabel('Valor')
axes[0].set_title('Métricas NDCG@k')
axes[0].set_xticks(x)
axes[0].set_xticklabels([k.upper() for k in ndcg_metrics])
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Gráfico de MRR
mrr_data = ['MRR']
mrr_values_test = [test_metrics['mrr']]
mrr_values_val = [val_metrics['mrr']]

axes[1].bar(['Teste'], mrr_values_test, alpha=0.8, label='Teste')
axes[1].bar(['Validação'], mrr_values_val, alpha=0.8, label='Validação')
axes[1].set_ylabel('Valor MRR')
axes[1].set_title('Mean Reciprocal Rank (MRR)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Análise de performance por query (top 5 e bottom 5)
per_query_ndcg = {}
for qid in detailed_test_eval['valid_query_ids']:
    qid_metrics = detailed_test_eval['per_query_metrics'][qid]
    per_query_ndcg[qid] = qid_metrics['ndcg@10']

# Ordenar queries por performance
sorted_queries = sorted(per_query_ndcg.items(), key=lambda x: x[1], reverse=True)

print(f"\n🏆 Top 5 Queries (NDCG@10):")
for i, (qid, ndcg) in enumerate(sorted_queries[:5]):
    print(f"  {i+1}. Query {qid}: {ndcg:.4f}")

print(f"\n📉 Bottom 5 Queries (NDCG@10):")
for i, (qid, ndcg) in enumerate(sorted_queries[-5:]):
    print(f"  {i+1}. Query {qid}: {ndcg:.4f}")

## 9. Logging no MLflow

In [ ]:
# Iniciar run no MLflow
run_name = f"notebook_demo_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
run_id = mlflow_manager.start_run(
    run_name=run_name,
    tags={
        "source": "jupyter_notebook",
        "model_type": "LambdaMART",
        "data_format": "SVM-Rank"
    }
)

print(f"🚀 MLflow run iniciado: {run_id}")

try:
    # 1. Log dos hiperparâmetros do modelo
    print("📝 Logging hiperparâmetros...")
    mlflow_manager.log_model_parameters(model_params)
    
    # 2. Log das configurações de treinamento
    mlflow_manager.log_model_parameters({
        f"training_{k}": v for k, v in training_config.items()
    })
    
    # 3. Log das informações dos datasets
    print("📊 Logging informações dos datasets...")
    mlflow_manager.log_dataset_info(dataset_info['train'], dataset_info['test'])
    
    # 4. Log das métricas de avaliação
    print("📈 Logging métricas de avaliação...")
    mlflow_manager.log_evaluation_metrics(test_metrics, prefix="test_")
    mlflow_manager.log_evaluation_metrics(val_metrics, prefix="val_")
    
    # 5. Log do histórico de treinamento
    if training_results.get('training_history'):
        print("📉 Logging histórico de treinamento...")
        mlflow_manager.log_training_metrics(training_results['training_history'])
    
    # 6. Log da importância das features
    print("🎯 Logging importância das features...")
    feature_importance = model.get_feature_importance()
    feature_names = [f"feature_{i}" for i in range(len(feature_importance))]
    mlflow_manager.log_feature_importance(feature_importance, feature_names)
    
    # 7. Log das predições
    print("🔮 Logging predições...")
    mlflow_manager.log_predictions(test_predictions, test_labels, test_qids, "test")
    mlflow_manager.log_predictions(val_predictions, val_lab, val_q, "validation")
    
    # 8. Log do modelo treinado
    print("🤖 Logging modelo...")
    mlflow_manager.log_model_artifact(model, "lambdamart_model")
    
    # 9. Criar e logar gráfico de resumo das métricas
    print("📊 Logging visualizações...")
    mlflow_manager.create_metrics_summary_plot(test_metrics)
    
    # 10. Log de métricas adicionais
    additional_metrics = {
        "training_duration_seconds": training_duration,
        "best_iteration": training_results['best_iteration'],
        "num_features": model_summary['num_features'],
        "num_trees": model_summary['num_trees'],
        "test_queries_evaluated": detailed_test_eval['num_valid_queries'],
        "val_queries_evaluated": len(np.unique(val_q))
    }
    mlflow_manager.log_evaluation_metrics(additional_metrics, prefix="meta_")
    
    print("✅ Todos os dados foram registrados no MLflow com sucesso!")
    
    # Exibir informações do run
    print(f"\n📋 Informações do Run:")
    print(f"  Run ID: {run_id}")
    print(f"  Run Name: {run_name}")
    print(f"  Experiment: {experiment_name}")
    
    # Mostrar métricas principais
    print(f"\n🎯 Métricas Principais:")
    print(f"  NDCG@1:  {test_metrics['ndcg@1']:.4f}")
    print(f"  NDCG@3:  {test_metrics['ndcg@3']:.4f}")
    print(f"  NDCG@5:  {test_metrics['ndcg@5']:.4f}")
    print(f"  NDCG@10: {test_metrics['ndcg@10']:.4f}")
    print(f"  MRR:     {test_metrics['mrr']:.4f}")
    
except Exception as e:
    print(f"❌ Erro durante o logging: {e}")
    
finally:
    # Finalizar o run
    mlflow_manager.end_run()
    print(f"🏁 Run do MLflow finalizado")
    
# Verificar runs no experimento
print(f"\n📊 Resumo do Experimento:")
try:
    runs_df = mlflow_manager.get_experiment_runs()
    if not runs_df.empty:
        print(f"Total de runs: {len(runs_df)}")
        
        # Mostrar os últimos 3 runs com métricas principais
        latest_runs = runs_df.sort_values('start_time', ascending=False).head(3)
        
        metrics_cols = ['metrics.test_ndcg@10', 'metrics.test_mrr', 'status']
        available_cols = [col for col in metrics_cols if col in latest_runs.columns]
        
        if available_cols:
            print("\nÚltimos runs:")
            display_cols = ['run_name'] + available_cols
            print(latest_runs[display_cols].to_string(index=False))
        
except Exception as e:
    print(f"Não foi possível recuperar informações dos runs: {e}")

## 10. Visualização dos Resultados

In [ ]:
# Criar visualizações abrangentes dos resultados

# 1. Dashboard de Métricas
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
fig.suptitle('Dashboard de Resultados do Modelo LambdaMART', fontsize=16, fontweight='bold')

# 1.1 Comparação NDCG@k
ndcg_metrics = [k for k in test_metrics.keys() if k.startswith('ndcg')]
test_ndcg = [test_metrics[k] for k in ndcg_metrics]
val_ndcg = [val_metrics[k] for k in ndcg_metrics]

x = np.arange(len(ndcg_metrics))
width = 0.35

axes[0,0].bar(x - width/2, test_ndcg, width, label='Teste', alpha=0.8, color='skyblue')
axes[0,0].bar(x + width/2, val_ndcg, width, label='Validação', alpha=0.8, color='lightcoral')
axes[0,0].set_xlabel('Métrica')
axes[0,0].set_ylabel('Valor')
axes[0,0].set_title('Comparação NDCG@k')
axes[0,0].set_xticks(x)
axes[0,0].set_xticklabels([k.replace('ndcg@', '@') for k in ndcg_metrics])
axes[0,0].legend()
axes[0,0].grid(True, alpha=0.3)

# 1.2 MRR Comparison
mrr_values = [test_metrics['mrr'], val_metrics['mrr']]
mrr_labels = ['Teste', 'Validação']
colors = ['skyblue', 'lightcoral']

bars = axes[0,1].bar(mrr_labels, mrr_values, color=colors, alpha=0.8)
axes[0,1].set_ylabel('MRR')
axes[0,1].set_title('Mean Reciprocal Rank')
axes[0,1].grid(True, alpha=0.3)

# Adicionar valores nas barras
for bar, value in zip(bars, mrr_values):
    height = bar.get_height()
    axes[0,1].text(bar.get_x() + bar.get_width()/2., height,
                   f'{value:.4f}', ha='center', va='bottom', fontweight='bold')

# 1.3 Feature Importance (Top 10)
feature_importance = model.get_feature_importance()
top_features_idx = np.argsort(feature_importance)[-10:][::-1]
top_importance = feature_importance[top_features_idx]
top_feature_names = [f'Feature {i}' for i in top_features_idx]

axes[0,2].barh(range(len(top_importance)), top_importance, color='lightgreen', alpha=0.8)
axes[0,2].set_yticks(range(len(top_importance)))
axes[0,2].set_yticklabels(top_feature_names)
axes[0,2].set_xlabel('Importância')
axes[0,2].set_title('Top 10 Features Mais Importantes')
axes[0,2].grid(True, alpha=0.3)

# 1.4 Distribuição de Predições vs Labels
axes[1,0].scatter(test_labels, test_predictions, alpha=0.6, s=20, color='purple')
axes[1,0].set_xlabel('Labels Verdadeiros')
axes[1,0].set_ylabel('Predições')
axes[1,0].set_title('Predições vs Labels Verdadeiros')
axes[1,0].plot([test_labels.min(), test_labels.max()], 
               [test_labels.min(), test_labels.max()], 'r--', alpha=0.8, linewidth=2)
axes[1,0].grid(True, alpha=0.3)

# 1.5 Distribuição de Performance por Query
query_ndcg_values = list(per_query_ndcg.values())
axes[1,1].hist(query_ndcg_values, bins=20, alpha=0.7, color='orange', edgecolor='black')
axes[1,1].set_xlabel('NDCG@10')
axes[1,1].set_ylabel('Número de Queries')
axes[1,1].set_title('Distribuição de NDCG@10 por Query')
axes[1,1].axvline(np.mean(query_ndcg_values), color='red', linestyle='--', 
                  label=f'Média: {np.mean(query_ndcg_values):.3f}')
axes[1,1].legend()
axes[1,1].grid(True, alpha=0.3)

# 1.6 Curva de Treinamento (se disponível)
if training_results.get('training_history'):
    history = training_results['training_history']
    
    # Encontrar métricas de treino e validação
    train_metrics = None
    val_metrics_hist = None
    
    for dataset_name, metrics_dict in history.items():
        if 'train' in dataset_name.lower():
            train_metrics = metrics_dict
        elif 'valid' in dataset_name.lower():
            val_metrics_hist = metrics_dict
    
    if train_metrics and 'ndcg' in train_metrics:
        iterations = range(len(train_metrics['ndcg']))
        axes[1,2].plot(iterations, train_metrics['ndcg'], label='Treino', color='blue')
        
        if val_metrics_hist and 'ndcg' in val_metrics_hist:
            axes[1,2].plot(iterations, val_metrics_hist['ndcg'], label='Validação', color='red')
        
        axes[1,2].set_xlabel('Iteração')
        axes[1,2].set_ylabel('NDCG')
        axes[1,2].set_title('Curva de Treinamento')
        axes[1,2].legend()
        axes[1,2].grid(True, alpha=0.3)
    else:
        axes[1,2].text(0.5, 0.5, 'Histórico de\nTreinamento\nNão Disponível', 
                       ha='center', va='center', transform=axes[1,2].transAxes,
                       fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))
        axes[1,2].set_title('Curva de Treinamento')
else:
    axes[1,2].text(0.5, 0.5, 'Histórico de\nTreinamento\nNão Disponível', 
                   ha='center', va='center', transform=axes[1,2].transAxes,
                   fontsize=12, bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))
    axes[1,2].set_title('Curva de Treinamento')

plt.tight_layout()
plt.show()

# 2. Análise Detalhada de Ranking para Queries Específicas
print("\\n" + "="*80)
print("📊 ANÁLISE DETALHADA DE RANKING")
print("="*80)

# Selecionar 3 queries para análise detalhada
sample_queries = sorted_queries[:3]  # Top 3 queries

for i, (qid, ndcg_score) in enumerate(sample_queries):
    print(f"\\n🔍 Query {qid} (NDCG@10: {ndcg_score:.4f})")
    print("-" * 50)
    
    # Obter dados da query
    query_mask = test_qids == qid
    query_labels = test_labels[query_mask]
    query_preds = test_predictions[query_mask]
    
    # Ordenar por predições
    sorted_indices = np.argsort(query_preds)[::-1]
    sorted_labels = query_labels[sorted_indices]
    sorted_preds = query_preds[sorted_indices]
    
    # Mostrar ranking
    print("Pos | Label | Predição | Relevante?")
    print("-" * 35)
    for pos, (label, pred) in enumerate(zip(sorted_labels[:10], sorted_preds[:10])):
        relevant = "✓" if label > 0 else "✗"
        print(f"{pos+1:3d} | {label:5.0f} | {pred:8.4f} | {relevant:>9s}")

# 3. Resumo Final
print("\\n" + "="*80)
print("📋 RESUMO FINAL DO EXPERIMENTO")
print("="*80)

summary_data = {
    'Métrica': ['NDCG@1', 'NDCG@3', 'NDCG@5', 'NDCG@10', 'MRR'],
    'Teste': [test_metrics['ndcg@1'], test_metrics['ndcg@3'], 
              test_metrics['ndcg@5'], test_metrics['ndcg@10'], test_metrics['mrr']],
    'Validação': [val_metrics['ndcg@1'], val_metrics['ndcg@3'], 
                  val_metrics['ndcg@5'], val_metrics['ndcg@10'], val_metrics['mrr']]
}

summary_df = pd.DataFrame(summary_data)
print("\\n🎯 Métricas Finais:")
print(summary_df.round(4).to_string(index=False))

print(f"\\n⚙️ Configuração do Modelo:")
print(f"  Learning Rate: {model_params['learning_rate']}")
print(f"  Num Leaves: {model_params['num_leaves']}")
print(f"  Num Boost Rounds: {training_config['num_boost_round']}")
print(f"  Early Stopping: {training_config['early_stopping_rounds']}")

print(f"\\n📊 Estatísticas do Dataset:")
print(f"  Treino: {len(train_features)} amostras, {len(np.unique(train_qids))} queries")
print(f"  Teste: {len(test_features)} amostras, {len(np.unique(test_qids))} queries")

print(f"\\n🕐 Tempo de Treinamento: {training_duration:.2f} segundos")
print(f"🌟 Melhor Iteração: {training_results['best_iteration']}")

print("\\n✅ Experimento concluído com sucesso!")

## Conclusão

Este notebook demonstrou como utilizar o framework LambdaMART desenvolvido para:

### ✅ Funcionalidades Implementadas

1. **Carregamento de Dados**: Leitura de arquivos SVM-Rank (.txt) com parser customizado
2. **Modelo LambdaMART**: Implementação usando LightGBM com configurações otimizadas
3. **Métricas de Avaliação**: Cálculo de NDCG@1,3,5,10 e MRR
4. **Integração MLflow**: Tracking completo de experimentos, hiperparâmetros e artefatos
5. **Visualizações**: Dashboards para análise de performance e resultados

### 📊 Resultados Obtidos

As métricas calculadas mostram a eficácia do modelo treinado:
- **NDCG@10**: Indica a qualidade do ranking nos top 10 resultados
- **MRR**: Mede a posição do primeiro documento relevante
- **Comparação Treino/Validação**: Verifica overfitting

### 🚀 Próximos Passos

1. **Otimização de Hiperparâmetros**: Usar grid search ou otimização bayesiana
2. **Feature Engineering**: Criar features mais informativas
3. **Ensemble Methods**: Combinar múltiplos modelos
4. **Cross-Validation**: Validação mais robusta com k-fold

### 💡 Como Usar com Seus Dados

1. Coloque seus arquivos .txt no formato SVM-Rank em `../data/train/` e `../data/test/`
2. Ajuste os hiperparâmetros na seção 5
3. Execute o notebook completo
4. Verifique os resultados no MLflow UI: `mlflow ui` no terminal

### 📚 Documentação

- **Formato SVM-Rank**: `<label> qid:<query_id> <feature_id>:<value> ...`
- **MLflow UI**: Acesse `http://localhost:5000` após executar `mlflow ui`
- **Código Fonte**: Verifique os módulos em `../src/` para customizações